# CMIP masked-point prediction — EXP3

to run this notebook with a job :

qsub -v NUM_SAMPLE=500000,OUTPUT_PREFIX=exp3_ns500000_CNN_AEh_central_1 run_exp3_mask_pipeline.pbs

### Data preprocessing

**Library import**

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
import json
from pathlib import Path

In [ ]:
ae_train_configs = {
    "AEh": ["historical"],
    "AEhs2": ["historical", "ssp245"],
    "AEhs2s3": ["historical", "ssp245", "ssp370"],
    "AEall": ["historical", "ssp245", "ssp370", "ssp585"]
}

EXPERIENCE 3 HYPERPARAMETERS :

In [ ]:
setup_name = "exp3"

In [ ]:
num_sample = 10000
chosen_autoencoder_type = "CNN"  # choose between "MLP" and "CNN"
ae_setup_name = "AEh"  # choose one of the keys above
mask_strategy = "central_6"  # options: central_1, central_6, checkerboard, hidden_bottom, keep_central_6, keep_central_1
val_fraction = 0.05
test_fraction = 0.15

**Data Loading**

You have to run a PBS job to create the data loaded in the next cell. In the pbs file you can choose the following parameters :

- --max-abs-lat value \ (recommanded : 30)
- --patch-size-km value \ (recommanded : 1000)
- --time-stride value \ (recommanded : 24)
- --max-samples-per-climate value \ (recommanded : 1000)
- --random-seed value \ (recommanded : 42)

to run the PBS job use the following command in /glade/u/home/tsalin/CMIP/jobs: 

qsub run_build_multivariate_samples.pbs

to follow what's going on : 

qstat -u tsalin

In [ ]:
precomputed_dir = Path(f"/glade/derecho/scratch/tsalin/CMIP/derived/multivariate_samples_optimized_NG_v{num_sample}")
if not precomputed_dir.exists():
    raise FileNotFoundError(f"Precomputed data directory not found: {precomputed_dir}")

with open(precomputed_dir / "run_config.json", "r", encoding="utf-8") as f:
    run_cfg = json.load(f)

climate_order = list(run_cfg["climate_order"])
climate_colors = dict(run_cfg["climate_colors"])
selected_variables_full = list(run_cfg["selected_variables"])
selected_variables = list(selected_variables_full)
max_abs_lat = float(run_cfg["max_abs_lat"])
patch_size_km = float(run_cfg["patch_size_km"])
time_stride = int(run_cfg["time_stride"])
max_samples_per_climate = int(run_cfg["max_samples_per_climate"])
random_seed = int(run_cfg["random_seed"])
n_lat = int(run_cfg["n_lat"])
n_lon = int(run_cfg["n_lon"])
grid_points_per_patch = int(run_cfg["grid_points_per_patch"])
n_patches = int(run_cfg["n_patches"])

features_by_climate_full = {
    c: np.load(precomputed_dir / f"features_{c}.npy")
    for c in climate_order
}
metadata_by_climate = {
    c: pd.read_csv(precomputed_dir / f"metadata_{c}.csv")
    for c in climate_order
}

n_variables_full = len(selected_variables_full)
expected_dim_full = n_variables_full * grid_points_per_patch

for c in climate_order:
    X_full = features_by_climate_full[c]
    if X_full.shape[1] != expected_dim_full:
        raise ValueError(
            f"Unexpected feature dimension for {c}: got {X_full.shape[1]}, expected {expected_dim_full}."
        )

# In the masked AE task, all variables remain in the sample.
# The label is not a separate climate variable anymore: the AE is trained to reconstruct only visible points.

sample_count_df = pd.read_csv(precomputed_dir / "sample_count.csv").set_index("scenario")
pre_sampling_df = pd.read_csv(precomputed_dir / "pre_sampling_df.csv").set_index("scenario")
patch_catalog = pd.read_csv(precomputed_dir / "patch_catalog.csv")
sampling_diagnostics_df = pd.read_csv(precomputed_dir / "sampling_diagnostics.csv").set_index("scenario")
nan_summary_by_variable_df = pd.read_csv(precomputed_dir / "nan_summary_by_variable.csv")

display(sample_count_df)
display(pre_sampling_df)
display(sampling_diagnostics_df)
print(f"Variables used by the masked AE: {selected_variables_full}")
print(f"Full sample shape: {expected_dim_full} = {n_variables_full} variables x {grid_points_per_patch} grid points")

In [ ]:
# Backward-compatible aliases used later in the notebook
climate_order = list(climate_order)
climate_colors = dict(climate_colors)
sample_count_df = sample_count_df.copy()
pre_sampling_df = pre_sampling_df.copy()
patch_catalog = patch_catalog.copy()
sampling_diagnostics_df = sampling_diagnostics_df.copy()
nan_summary_by_variable_df = nan_summary_by_variable_df.copy()

**Latitude Band, 1000-km Patches, and Multivariate Samples**

This step loads one unified multivariate dataset used by all downstream analyses.

Each sample is a geographic patch at one time step, flattened as:

```text
n_variables × grid_points_per_patch = 6 × 70 = 420 features
```

Unlike the previous variable-prediction version, no variable is removed from the input. The task is now a masked-point reconstruction task: some geographic points are hidden, their values are replaced by `0.0`, and an explicit binary mask channel is appended to the AE input.

Masked-point task and rigorous preprocessing setup

In [ ]:
# Mask definition.
# Feature layout: variables are contiguous blocks of grid_points_per_patch.
# Feature index for variable v and point p is: v * grid_points_per_patch + p.

masked_point_indices = None  # set manually to override mask_strategy, e.g. np.array([31, 32, 38, 39])
mask_fill_value = 0.0

if masked_point_indices is None:
    if mask_strategy == "central_1":
        masked_point_indices = np.array([
            4 * n_lon + 3,
        ], dtype=int)

    elif mask_strategy == "central_6":
        center_i = n_lat // 2
        center_j = n_lon // 2
        masked_point_indices = np.array([
            (center_i - 1) * n_lon + (center_j - 1),
            center_i * n_lon + (center_j - 1),
            (center_i - 1) * n_lon + center_j,
            center_i * n_lon + center_j,
            (center_i - 1) * n_lon + (center_j + 1),
            center_i * n_lon + (center_j + 1),
        ], dtype=int)

    elif mask_strategy == "checkerboard":
        masked_point_indices = np.array([
            i * n_lon + j
            for i in range(n_lat)
            for j in range(n_lon)
            if (i + j) % 2 == 0
        ], dtype=int)
    
    elif mask_strategy == "hidden_bottom":
        masked_point_indices = np.array([
            i * n_lon + j
            for i in range(n_lat)
            for j in range(n_lon)
            if i < n_lat // 2
        ], dtype=int)

    elif mask_strategy == "keep_central_6":
        visible_point_indices = np.array([
            5 * n_lon + 2,
            5 * n_lon + 3,
            5 * n_lon + 4,
            4 * n_lon + 2,
            4 * n_lon + 3,
            4 * n_lon + 4,
        ], dtype=int)
        masked_point_indices = np.setdiff1d(np.arange(grid_points_per_patch), visible_point_indices)
    
    elif mask_strategy == "keep_central_1":
        visible_point_indices = np.array([
            4 * n_lon + 3,
        ], dtype=int)
        masked_point_indices = np.setdiff1d(np.arange(grid_points_per_patch), visible_point_indices)

    else:
        raise ValueError(f"Unknown mask_strategy={mask_strategy!r}.")
else:
    masked_point_indices = np.asarray(masked_point_indices, dtype=int)

if masked_point_indices.ndim != 1:
    raise ValueError("masked_point_indices must be a 1D array of grid-point indices.")
if len(masked_point_indices) == 0:
    raise ValueError("At least one grid point must be masked.")
if masked_point_indices.min() < 0 or masked_point_indices.max() >= grid_points_per_patch:
    raise ValueError(
        f"masked_point_indices must be between 0 and {grid_points_per_patch - 1}. "
        f"Got range [{masked_point_indices.min()}, {masked_point_indices.max()}]."
    )

masked_point_indices = np.unique(masked_point_indices.astype(int))
visible_point_indices = np.setdiff1d(np.arange(grid_points_per_patch), masked_point_indices)

masked_feature_indices = np.concatenate([
    v * grid_points_per_patch + masked_point_indices
    for v in range(n_variables_full)
]).astype(int)
visible_feature_indices = np.setdiff1d(np.arange(expected_dim_full), masked_feature_indices)

# Binary reconstruction weights: 1 for visible original variables, 0 for masked original variables.
reconstruction_weight_vector = np.zeros(expected_dim_full, dtype=np.float32)
reconstruction_weight_vector[visible_feature_indices] = 1.0

# Binary mask channel appended to the model input.
# 1 = visible value, 0 = masked value.
input_mask_vector = reconstruction_weight_vector.copy()

print(f"Mask strategy: {mask_strategy}")
print(f"Masked point indices ({len(masked_point_indices)} / {grid_points_per_patch}): {masked_point_indices}")
print(f"Visible point count: {len(visible_point_indices)} / {grid_points_per_patch}")
print(f"Masked feature count: {len(masked_feature_indices)} / {expected_dim_full}")
print(f"Visible feature count: {len(visible_feature_indices)} / {expected_dim_full}")

In [ ]:
def plot_patch_mask(masked_point_indices, patch_id=0):
    """Visualize the geographic layout of the masked grid points for one patch."""
    masked_point_indices_arr = np.asarray(masked_point_indices, dtype=int)
    patch_info = patch_catalog.loc[patch_catalog["patch_id"] == patch_id].iloc[0]

    patch_n_lat = int(patch_info["lat_stop_idx"] - patch_info["lat_start_idx"])
    patch_n_lon = int(patch_info["lon_stop_idx"] - patch_info["lon_start_idx"])
    if patch_n_lat * patch_n_lon != grid_points_per_patch:
        raise ValueError(
            f"Patch shape mismatch: {patch_n_lat} x {patch_n_lon} != {grid_points_per_patch} points."
        )

    lats = np.linspace(patch_info["lat_start"], patch_info["lat_stop"], patch_n_lat)
    lons = np.linspace(patch_info["lon_start"], patch_info["lon_stop"], patch_n_lon)
    lon_grid, lat_grid = np.meshgrid(lons, lats)
    flat_lons = lon_grid.ravel()
    flat_lats = lat_grid.ravel()

    is_masked = np.zeros(grid_points_per_patch, dtype=bool)
    is_masked[masked_point_indices_arr] = True

    fig, ax = plt.subplots(figsize=(7, 5.5), constrained_layout=True)
    ax.scatter(
        flat_lons[~is_masked],
        flat_lats[~is_masked],
        s=65,
        color="#4C72B0",
        alpha=0.8,
        edgecolor="white",
        linewidth=0.6,
        label="Visible points",
        zorder=2,
    )
    ax.scatter(
        flat_lons[is_masked],
        flat_lats[is_masked],
        s=170,
        marker="X",
        color="#C44E52",
        edgecolor="black",
        linewidth=1.0,
        label="Masked points",
        zorder=5,
    )

    for idx in range(grid_points_per_patch):
        ax.text(
            flat_lons[idx],
            flat_lats[idx],
            str(idx),
            ha="center",
            va="center",
            fontsize=7,
            color="white" if is_masked[idx] else "black",
            fontweight="bold" if is_masked[idx] else "normal",
            zorder=6,
        )

    ax.set_title(f"Masked grid points for patch {patch_id}")
    ax.set_xlabel("Longitude")
    ax.set_ylabel("Latitude")
    ax.grid(alpha=0.25)
    ax.legend(frameon=True)
    ax.set_aspect("equal", adjustable="box")
    plt.show()


plot_patch_mask(masked_point_indices, patch_id=0)

# Third Experiment - Autoencoders Trained on Increasing Climate Diversity

In this third experiment, we want to study the distribution shift of our data across climates inside an autoencoder (AE). So we focus on the latent representation of an AE trained on some climates.

We retrieve the patchs used in the second experiment. We randomly split these samples in train/val/test datasets for each climate.
We then train the AE on one of these configurations (using the train and validation sets) :
- historical climate
- historical and ssp245 climates
- historical, ssp245 and ssp370 climates
- all climates

Then we evaluate the reconstruction quality of the AE on all climates (using the test sets)

### AE construction and training


Choose the training setup at the top of this section. The AE is trained only on the train split of the climates listed in the selected setup, while all climates are kept for evaluation.


Climates trained on :

In [ ]:
ae_train_climates = ae_train_configs[ae_setup_name]

Hyperparameters of the AE

In [ ]:
ae_latent_dim = 64
ae_batch_size = 128
ae_learning_rate = 1e-3
ae_n_epochs = 30
ae_patience = 5
ae_weight_decay = 1e-5
ae_checkpoint_path = Path("/glade/u/home/tsalin/CMIP/model_evaluation/exp3/ae_mask_checkpoint.pt")
ae_checkpoint_freq = 1  # Save checkpoint every N epochs (1 = every epoch)

Utilities :

In [ ]:
torch.manual_seed(random_seed)
np.random.seed(random_seed)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)
print("Selected setup:", ae_setup_name)
print("Training climates:", ", ".join(ae_train_climates))

climate_to_idx = {c: i for i, c in enumerate(climate_order)}
idx_to_climate = {i: c for c, i in climate_to_idx.items()}

Train/validation/test split before standardization

In [ ]:
def build_split_indices(data_by_climate, val_fraction=0.10, test_fraction=0.20, seed=42):
    # Build train/val/test split indices for each climate before any standardization.
    # This prevents validation/test information from entering the preprocessing step.
    split_indices = {}
    rng = np.random.default_rng(seed)
    for climate, X in data_by_climate.items():
        n = X.shape[0]
        indices = np.arange(n)
        rng.shuffle(indices)

        n_test = max(1, int(round(test_fraction * n)))
        n_val = max(1, int(round(val_fraction * n)))
        n_train = max(1, n - n_val - n_test)

        train_idx = indices[:n_train]
        val_idx = indices[n_train:n_train + n_val]
        test_idx = indices[n_train + n_val:]

        split_indices[climate] = {
            "train": train_idx,
            "val": val_idx,
            "test": test_idx,
        }
    return split_indices


ae_split_indices = build_split_indices(
    features_by_climate_full,
    val_fraction=val_fraction,
    test_fraction=test_fraction,
    seed=random_seed,
)

# ── RAM optimisation  ───────
_eval_only_climates = [c for c in climate_order if c not in ae_train_climates]
if _eval_only_climates:
    for c in _eval_only_climates:
        _idx = ae_split_indices[c]["test"]
        features_by_climate_full[c] = features_by_climate_full[c][_idx]
        metadata_by_climate[c]      = metadata_by_climate[c].iloc[_idx].reset_index(drop=True)
        ae_split_indices[c] = {
            "train": np.array([], dtype=np.intp),
            "val":   np.array([], dtype=np.intp),
            "test":  np.arange(len(_idx), dtype=np.intp),
        }
    print(f"RAM opt : {_eval_only_climates} truncated at {len(_idx)} samples (test split).")
    del _idx
del _eval_only_climates
# ── ENd of RAM optimisation ──────────────────────────────────────────────────────

# Rigorous variable-wise standardization:
# - split is already built before standardization;
# - fit statistics only on historical train samples;
# - fit statistics only on visible points;
# - estimate one mean/std per physical variable;
# - apply that variable-wise mean/std to all points of that variable;
# - replace masked values by mask_fill_value after standardization.

variable_means = np.zeros(n_variables_full, dtype=np.float64)
variable_stds = np.ones(n_variables_full, dtype=np.float64)

visible_point_indices = np.setdiff1d(
    np.arange(grid_points_per_patch, dtype=int),
    masked_point_indices,
    assume_unique=False,
)

for var_idx, var_name in enumerate(selected_variables_full):
    visible_cols_for_var = (
        var_idx * grid_points_per_patch + visible_point_indices
    )

    train_idx = ae_split_indices["historical"]["train"]

    train_visible_values = features_by_climate_full["historical"][
        train_idx
    ][:, visible_cols_for_var].reshape(-1)

    train_visible_values = train_visible_values[
        np.isfinite(train_visible_values)
    ]

    if train_visible_values.size == 0:
        raise ValueError(
            f"No finite historical train visible values found for variable {var_name!r}."
        )

    if var_name == "pr":
        train_visible_values = np.log1p(train_visible_values * 86400)

    variable_means[var_idx] = np.mean(train_visible_values)
    variable_stds[var_idx] = np.std(train_visible_values)

    if not np.isfinite(variable_stds[var_idx]) or variable_stds[var_idx] <= 0:
        variable_stds[var_idx] = 1.0


scaled_target_by_climate = {}
scaled_features_by_climate = {}
reconstruction_weights_by_climate = {}

for climate in climate_order:
    X_raw = np.asarray(features_by_climate_full[climate], dtype=np.float32)

    X_scaled_full = np.empty_like(X_raw, dtype=np.float32)

    for var_idx, var_name in enumerate(selected_variables_full):
        cols_for_var = slice(
            var_idx * grid_points_per_patch,
            (var_idx + 1) * grid_points_per_patch,
        )
        values = X_raw[:, cols_for_var]
        if var_name == "pr":
            values = np.log1p(values * 86400)
        X_scaled_full[:, cols_for_var] = (
            values - variable_means[var_idx]
        ) / variable_stds[var_idx]

    # The AE input contains standardized values with hidden point values replaced by mask_fill_value.
    X_masked = X_scaled_full.copy()
    X_masked[:, masked_feature_indices] = mask_fill_value

    # Append an explicit mask channel.
    # Here input_mask_vector should have length expected_dim_full:
    #   1 for visible features, 0 for masked features
    mask_channel = np.broadcast_to(
        input_mask_vector,
        X_masked.shape
    ).astype(np.float32)

    X_model_input = np.concatenate(
        [X_masked, mask_channel],
        axis=1
    ).astype(np.float32)

    scaled_target_by_climate[climate] = X_scaled_full
    scaled_features_by_climate[climate] = X_model_input

    reconstruction_weights_by_climate[climate] = np.broadcast_to(
        reconstruction_weight_vector,
        X_scaled_full.shape,
    ).astype(np.float32)


standardization_summary_df = pd.DataFrame({
    "variable": selected_variables_full,
    "mean_fit_on_historical_train_visible_points": variable_means,
    "std_fit_on_historical_train_visible_points": variable_stds,
})

display(standardization_summary_df)

# Convenience names used by the model definitions.
model_input_dim = expected_dim_full * 2
reconstruction_output_dim = expected_dim_full
input_dim = model_input_dim

print("Scaler fitted on: historical train split, visible features only")
print("Model input dim:", model_input_dim)
print("Reconstruction output dim:", reconstruction_output_dim)
print("Example input shape:", next(iter(scaled_features_by_climate.values())).shape)
print("Example target shape:", next(iter(scaled_target_by_climate.values())).shape)

In [ ]:
# RAM Reduction
del features_by_climate_full

Different structures for the AE :

Structure inspired by CERA (CNN2D) - it uses the spatial structure of the samples

In [ ]:
input_dim = model_input_dim
CNN_hidden_dim_1 = max(256, min(1024, input_dim // 2))
CNN_hidden_dim_2 = max(128, min(512, input_dim // 8))


class CNNEncoder(nn.Module):
    def __init__(self, input_channels, spatial_shape, latent_dim):
        super().__init__()
        self.input_channels = input_channels
        self.spatial_shape = spatial_shape
        self.features = nn.Sequential(
            nn.Conv2d(input_channels, 32, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(32, 32, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(32, 64, kernel_size=3, stride=2, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
        )
        with torch.no_grad():
            dummy = torch.zeros(1, input_channels, *spatial_shape)
            feature_map = self.features(dummy)
            self.feature_shape = tuple(feature_map.shape[1:])
            self.flatten_dim = int(np.prod(self.feature_shape))
        self.projection = nn.Linear(self.flatten_dim, latent_dim)

    def forward(self, x):
        if x.ndim == 2:
            x = x.reshape(x.shape[0], self.input_channels, *self.spatial_shape)
        elif x.ndim != 4:
            raise ValueError("Expected a 2D flat batch or a 4D image batch.")
        x = self.features(x)
        x = torch.flatten(x, start_dim=1)
        return self.projection(x)


class CNNDecoder(nn.Module):
    def __init__(self, output_channels, spatial_shape, latent_dim, feature_shape):
        super().__init__()
        self.output_channels = output_channels
        self.spatial_shape = spatial_shape
        self.feature_shape = feature_shape
        self.project = nn.Sequential(
            nn.Linear(latent_dim, int(np.prod(feature_shape))),
            nn.ReLU(inplace=True),
        )
        self.refine = nn.Sequential(
            nn.Conv2d(feature_shape[0], 64, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.Upsample(size=spatial_shape, mode="bilinear", align_corners=False),
            nn.Conv2d(64, 32, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(32, 32, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(32, output_channels, kernel_size=3, padding=1),
        )

    def forward(self, z):
        x = self.project(z)
        x = x.reshape(z.shape[0], *self.feature_shape)
        x = self.refine(x)
        return torch.flatten(x, start_dim=1)


class CNNAutoEncoder(nn.Module):
    def __init__(self, input_dim, latent_dim, hidden_dims=None):
        super().__init__()
        _ = hidden_dims
        self.data_channels = n_variables_full
        self.mask_channels = n_variables_full
        self.input_channels = self.data_channels + self.mask_channels
        self.output_channels = self.data_channels
        self.spatial_shape = (n_lat, n_lon)

        expected_input_dim = self.input_channels * grid_points_per_patch
        expected_output_dim = self.output_channels * grid_points_per_patch
        if input_dim != expected_input_dim:
            raise ValueError(
                f"input_dim={input_dim} is incompatible with a CNN reshape using "
                f"{self.input_channels} input channels and {grid_points_per_patch} grid points per patch."
            )
        if reconstruction_output_dim != expected_output_dim:
            raise ValueError(
                f"reconstruction_output_dim={reconstruction_output_dim} is incompatible with "
                f"{self.output_channels} output channels and {grid_points_per_patch} grid points per patch."
            )

        self.encoder = CNNEncoder(self.input_channels, self.spatial_shape, latent_dim)
        self.decoder = CNNDecoder(self.output_channels, self.spatial_shape, latent_dim, self.encoder.feature_shape)

    def forward(self, x):
        z = self.encoder(x)
        x_hat = self.decoder(z)
        return x_hat, z

A more classical structure using only MLPs - it doesn't use the spatial structure but apparently it performs better

In [ ]:
input_dim = model_input_dim
MLP_hidden_dim_1 = max(256, min(1024, input_dim // 2))
MLP_hidden_dim_2 = max(128, min(512, input_dim // 8))


class MLPEncoder(nn.Module):
    def __init__(self, input_dim, latent_dim, hidden_dims=(512, 256)):
        super().__init__()
        h1, h2 = hidden_dims
        self.net = nn.Sequential(
            nn.Linear(input_dim, h1),
            nn.ReLU(),
            nn.Linear(h1, h2),
            nn.ReLU(),
            nn.Linear(h2, latent_dim),
        )

    def forward(self, x):
        return self.net(x)


class MLPDecoder(nn.Module):
    def __init__(self, latent_dim, output_dim, hidden_dims=(256, 512)):
        super().__init__()
        h1, h2 = hidden_dims
        self.net = nn.Sequential(
            nn.Linear(latent_dim, h1),
            nn.ReLU(),
            nn.Linear(h1, h2),
            nn.ReLU(),
            nn.Linear(h2, output_dim),
        )

    def forward(self, z):
        return self.net(z)


class MLPAutoEncoder(nn.Module):
    def __init__(self, input_dim, latent_dim, hidden_dims=(512, 256)):
        super().__init__()
        self.encoder = MLPEncoder(input_dim, latent_dim, hidden_dims=hidden_dims)
        decoder_hidden_dims = tuple(reversed(hidden_dims))
        self.decoder = MLPDecoder(latent_dim, reconstruction_output_dim, hidden_dims=decoder_hidden_dims)

    def forward(self, x):
        z = self.encoder(x)
        x_hat = self.decoder(z)
        return x_hat, z

Utilities

In [ ]:
def stack_climates(input_by_climate, target_by_climate, weight_by_climate, split_indices, climates, split):
    xs, targets, weights, ys, climate_names = [], [], [], [], []
    for climate in climates:
        idx = split_indices[climate][split]
        Xc = np.asarray(input_by_climate[climate][idx], dtype=np.float32)
        Tc = np.asarray(target_by_climate[climate][idx], dtype=np.float32)
        Wc = np.asarray(weight_by_climate[climate][idx], dtype=np.float32)
        xs.append(Xc)
        targets.append(Tc)
        weights.append(Wc)
        ys.append(np.full(len(idx), climate_to_idx[climate], dtype=np.int64))
        climate_names.extend([climate] * len(idx))
    X = np.vstack(xs)
    T = np.vstack(targets)
    W = np.vstack(weights)
    y = np.concatenate(ys)
    return X, T, W, y, np.array(climate_names)


def make_loader(X, target, weight, y, batch_size=128, shuffle=True):
    ds = TensorDataset(
        torch.tensor(X, dtype=torch.float32),
        torch.tensor(target, dtype=torch.float32),
        torch.tensor(weight, dtype=torch.float32),
        torch.tensor(y, dtype=torch.long),
    )
    return DataLoader(ds, batch_size=batch_size, shuffle=shuffle)


def masked_mse_loss(pred, target, weight, eps=1e-8):
    # MSE over visible original variables only. Masked points have weight 0.
    sq_error = (pred - target) ** 2
    return (sq_error * weight).sum() / (weight.sum() + eps)

Training function :

In [ ]:
def train_standard_autoencoder(train_climates, latent_dim=16, n_epochs=20, lr=1e-3, batch_size=128, weight_decay=1e-5, checkpoint_path=None, checkpoint_freq: int = 1):
    X_train, T_train, W_train, y_train, _ = stack_climates(
        scaled_features_by_climate,
        scaled_target_by_climate,
        reconstruction_weights_by_climate,
        ae_split_indices,
        train_climates,
        "train",
    )
    X_val, T_val, W_val, y_val, _ = stack_climates(
        scaled_features_by_climate,
        scaled_target_by_climate,
        reconstruction_weights_by_climate,
        ae_split_indices,
        train_climates,
        "val",
    )

    train_loader = make_loader(X_train, T_train, W_train, y_train, batch_size=batch_size, shuffle=True)
    val_loader = make_loader(X_val, T_val, W_val, y_val, batch_size=batch_size, shuffle=False)

    if chosen_autoencoder_type == "CNN":
        model = CNNAutoEncoder(input_dim=input_dim, latent_dim=latent_dim, hidden_dims=(CNN_hidden_dim_1, CNN_hidden_dim_2)).to(device)
    else:
        model = MLPAutoEncoder(input_dim=input_dim, latent_dim=latent_dim, hidden_dims=(MLP_hidden_dim_1, MLP_hidden_dim_2)).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)

    best_state = None
    best_val = np.inf
    best_epoch = 0
    wait = 0
    history = []

    start_epoch = 1
    if checkpoint_path is not None and Path(checkpoint_path).exists():
        _ckpt = torch.load(checkpoint_path, map_location=device)
        model.load_state_dict(_ckpt["model_state"])
        optimizer.load_state_dict(_ckpt["optimizer_state"])
        best_val = _ckpt["best_val"]
        best_epoch = _ckpt["best_epoch"]
        wait = _ckpt["wait"]
        best_state = _ckpt["best_state"]
        history = _ckpt["history"]
        start_epoch = _ckpt["epoch"] + 1
        print(f"[Checkpoint] Resumed from epoch {start_epoch} (Best val={best_val:.6g})")

    for epoch in range(start_epoch, n_epochs + 1):
        model.train()
        train_losses = []
        for xb, tb, wb, _ in train_loader:
            xb = xb.to(device, non_blocking=True)
            tb = tb.to(device, non_blocking=True)
            wb = wb.to(device, non_blocking=True)

            optimizer.zero_grad(set_to_none=True)
            x_hat, _ = model(xb)
            loss = masked_mse_loss(x_hat, tb, wb)
            loss.backward()
            optimizer.step()
            train_losses.append(loss.item())

        model.eval()
        val_losses = []
        with torch.no_grad():
            for xb, tb, wb, _ in val_loader:
                xb = xb.to(device, non_blocking=True)
                tb = tb.to(device, non_blocking=True)
                wb = wb.to(device, non_blocking=True)
                x_hat, _ = model(xb)
                val_losses.append(masked_mse_loss(x_hat, tb, wb).item())

        train_loss = float(np.mean(train_losses))
        val_loss = float(np.mean(val_losses))
        history.append({"epoch": epoch, "train_loss": train_loss, "val_loss": val_loss})

        if val_loss < best_val - 1e-6:
            best_val = val_loss
            best_epoch = epoch
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            wait = 0
        else:
            wait += 1
            if wait >= ae_patience:
                print(f"Early stopping at epoch {epoch} (best epoch {best_epoch}, val={best_val:.6f})")
                if checkpoint_path is not None:
                    torch.save({
                        "epoch": epoch,
                        "model_state": {k: v.detach().cpu().clone() for k, v in model.state_dict().items()},
                        "optimizer_state": optimizer.state_dict(),
                        "best_val": best_val,
                        "best_epoch": best_epoch,
                        "wait": wait,
                        "best_state": best_state,
                        "history": history,
                    }, checkpoint_path)
                break

        if epoch == 1 or epoch % 5 == 0:
            print(f"Epoch {epoch:03d} | train={train_loss:.6f} | val={val_loss:.6f}")

        if checkpoint_path is not None and epoch % checkpoint_freq == 0:
            torch.save({
                "epoch": epoch,
                "model_state": {k: v.detach().cpu().clone() for k, v in model.state_dict().items()},
                "optimizer_state": optimizer.state_dict(),
                "best_val": best_val,
                "best_epoch": best_epoch,
                "wait": wait,
                "best_state": best_state,
                "history": history,
            }, checkpoint_path)

    if best_state is not None:
        model.load_state_dict(best_state)

    return model, pd.DataFrame(history), best_epoch

Training and training curve

In [ ]:
ae_model, ae_history_df, ae_best_epoch = train_standard_autoencoder(
    train_climates=ae_train_climates,
    latent_dim=ae_latent_dim,
    n_epochs=ae_n_epochs,
    lr=ae_learning_rate,
    batch_size=ae_batch_size,
    weight_decay=ae_weight_decay,
    checkpoint_path=ae_checkpoint_path,
    checkpoint_freq=ae_checkpoint_freq,
)

print("Best epoch:", ae_best_epoch)
print("Selected setup:", ae_setup_name)

fig, ax = plt.subplots(figsize=(7, 4), constrained_layout=True)
ax.plot(ae_history_df["epoch"], ae_history_df["train_loss"], label="train")
ax.plot(ae_history_df["epoch"], ae_history_df["val_loss"], label="val")
ax.set_title(f"Training curve - {ae_setup_name}")
ax.set_xlabel("Epoch")
ax.set_ylabel("MSE loss")
ax.grid(alpha=0.25)
ax.legend()
plt.show()


### Evaluating reconstruction quality - EXTRACTION

The reconstruction is evaluated on the test split of every climate

Note that the metrics (rmse, mae, mse, r2) are computed from the destandardized multivariate test sets, that means that they are representing physical quantities.

We evaluate :

In [ ]:
component_order = {"reconstruction": 0}


# ---------------------------------------------------------------------
# Visible / masked reconstruction metadata
# ---------------------------------------------------------------------
visible_point_indices = np.setdiff1d(
    np.arange(grid_points_per_patch, dtype=int),
    masked_point_indices,
    assume_unique=False,
)

visible_feature_columns = np.setdiff1d(
    np.arange(expected_dim_full, dtype=int),
    masked_feature_indices,
    assume_unique=False,
)

reconstruction_labels_visible = [
    f"{var}@point_{point_idx}"
    for var in selected_variables_full
    for point_idx in visible_point_indices
]


def denormalize_full_features(
    X_normalized: np.ndarray,
    variable_means: np.ndarray,
    variable_stds: np.ndarray,
    n_points: int,
    n_variables: int,
) -> np.ndarray:
    X_denorm = np.asarray(X_normalized, dtype=np.float32).copy()

    for var_idx, var_name in enumerate(selected_variables_full):
        cols_for_var = slice(var_idx * n_points, (var_idx + 1) * n_points)
        values = X_denorm[:, cols_for_var] * variable_stds[var_idx] + variable_means[var_idx]
        if var_name == "pr":
            values = np.expm1(values)
        X_denorm[:, cols_for_var] = values

    return X_denorm


def _build_meta_df(climate, metadata, latent_dim):
    df = metadata.copy().reset_index(drop=True)
    df = df.drop(columns=["scenario"], errors="ignore")
    df.insert(0, "experiment", "CMIP_mask_exp3")
    df.insert(1, "component", "reconstruction")
    df.insert(2, "component_order", component_order["reconstruction"])
    df.insert(3, "scenario", climate)
    df.insert(4, "scenario_order", int(climate_order.index(climate)))
    df.insert(5, "sample_idx", np.arange(len(df)))
    df.insert(6, "latent_dim", int(latent_dim))
    return df


def _reconstruct_in_batches(model, X, batch_size=4096):
    x_hat_chunks = []
    z_chunks = []

    model.eval()

    with torch.inference_mode():
        for start in range(0, X.shape[0], batch_size):
            xb = torch.as_tensor(
                X[start:start + batch_size],
                dtype=torch.float32,
                device=device,
            )

            x_hat, z = model(xb)

            x_hat_chunks.append(x_hat.detach().cpu().numpy())
            z_chunks.append(z.detach().cpu().numpy())

    return (
        np.vstack(x_hat_chunks),
        np.vstack(z_chunks),
    )


# Accumulators
meta_dfs_recon = []
truth_arrays_recon, pred_arrays_recon = [], []

ae_model.eval()

for climate in climate_order:
    idx = ae_split_indices[climate]["test"]

    X = np.asarray(
        scaled_features_by_climate[climate][idx],
        dtype=np.float32,
    )

    X_true_scaled = np.asarray(
        scaled_target_by_climate[climate][idx],
        dtype=np.float32,
    )

    metadata = metadata_by_climate[climate].iloc[idx].reset_index(drop=True)

    X_hat_scaled, z_np = _reconstruct_in_batches(
        ae_model,
        X,
        batch_size=4096,
    )

    X_true_physical = denormalize_full_features(
        X_true_scaled,
        variable_means,
        variable_stds,
        n_points=grid_points_per_patch,
        n_variables=n_variables_full,
    )

    X_hat_physical = denormalize_full_features(
        X_hat_scaled,
        variable_means,
        variable_stds,
        n_points=grid_points_per_patch,
        n_variables=n_variables_full,
    )

    meta_dfs_recon.append(_build_meta_df(climate, metadata, z_np.shape[1]))
    truth_arrays_recon.append(X_true_physical[:, visible_feature_columns].astype(np.float32))
    pred_arrays_recon.append(X_hat_physical[:, visible_feature_columns].astype(np.float32))


reconstruction_payload = {
    "meta_reconstruction": pd.concat(meta_dfs_recon, ignore_index=True),
    "truth_reconstruction": np.concatenate(truth_arrays_recon, axis=0),
    "pred_reconstruction": np.concatenate(pred_arrays_recon, axis=0),
    "reconstruction_value_names": reconstruction_labels_visible,
    "masked_point_indices": masked_point_indices.tolist(),
    "masked_feature_columns": masked_feature_indices.tolist(),
    "visible_point_indices": visible_point_indices.tolist(),
    "visible_feature_columns": visible_feature_columns.tolist(),
}

display(reconstruction_payload["meta_reconstruction"].head())
print("truth_reconstruction shape:", reconstruction_payload["truth_reconstruction"].shape)


### Extraction of the latent representations

In [ ]:
def extract_latents_by_climate(model, data_by_climate, split_indices, split="test", batch_size=2048):
    # Extract latent representations for each climate and split using the trained autoencoder.
    model.eval()
    out = {}
    with torch.no_grad():
        for climate in climate_order:
            idx = split_indices[climate][split]
            X = np.asarray(data_by_climate[climate][idx], dtype=np.float32)
            latents = []
            for start in range(0, len(X), batch_size):
                X_t = torch.tensor(X[start:start + batch_size], dtype=torch.float32, device=device)
                latents.append(model.encoder(X_t).cpu().numpy())
            out[climate] = np.vstack(latents) if latents else np.empty((0, ae_latent_dim), dtype=np.float32)
    return out


latent_test_by_climate = extract_latents_by_climate(
    ae_model,
    scaled_features_by_climate,
    ae_split_indices,
    split="test",
)
latent_test_metadata_by_climate = {
    climate: metadata_by_climate[climate].iloc[ae_split_indices[climate]["test"]].reset_index(drop=True)
    for climate in climate_order
}

# Explicit aliases for downstream notebooks.
ae_latent_by_climate = latent_test_by_climate
ae_latent_test_metadata_by_climate = latent_test_metadata_by_climate

In [ ]:
# Delete the checkpoint file after training to save disk space and maintain consistency, if it exists.
if ae_checkpoint_path.exists():
    ae_checkpoint_path.unlink()
    print(f"[Checkpoint] {ae_checkpoint_path.name} deleted.")